### Temporal Feature Engineering

Transform historical sensor measurements into leakage-safe temporal features that may help predict Remaining Useful Life (RUL).

Load the data

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_dataset

train_df, test_df, rul_df = load_dataset("FD001")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)



The original FD001 dataset is loaded using the reusable project data loader.

It starts from the same sensor observations used in the previous.

Add the RUL target

In [4]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"].transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)

In [5]:
RUL_CAP = 125

train_df["max_cycle"] = (
    train_df.groupby("unit")["cycle"].transform("max")
)

train_df["RUL_raw"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df["RUL"] = (
    train_df["RUL_raw"].clip(upper=RUL_CAP)
)


The RUL_Engineenring RUL definition is reused so that temporal features can be evaluated against the same target.

The target itself is not being redesigned in Phase 2.

Identify sensors

In [6]:
sensor_cols = [
    f"sensor_{i}"
    for i in range(1, 22)
]

print("Number of sensors:", len(sensor_cols))

Number of sensors: 21




The 21 sensor measurements are used as the starting point for temporal feature engineering.

Sort the data

In [7]:
train_df = train_df.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)

In [9]:
train_df[["unit", "cycle"]].head(10)

,unit,cycle
0,1,1
1,1,2
2,1,3
3,1,4
4,1,5
5,1,6
6,1,7
7,1,8
8,1,9
9,1,10



The data is sorted by engine and cycle so that all temporal features are calculated in the correct historical order.

In [10]:
raw_features = train_df[
    ["unit", "cycle"] + sensor_cols
].copy()

print(raw_features.shape)

(20631, 23)


#### Raw Sensor Baseline

The original sensor measurements form the baseline feature representation.

Temporal features will later be compared against this baseline.

In [11]:
train_df["sensor_2_lag1"] = (
    train_df.groupby("unit")["sensor_2"]
    .shift(1)
)

In [12]:
train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_lag1"]
].head(10)

,unit,cycle,sensor_2,sensor_2_lag1
0,1,1,641.82,NaN
1,1,2,642.15,641.82
2,1,3,642.35,642.15
3,1,4,642.35,642.35
4,1,5,642.37,642.35
5,1,6,642.10,642.37
6,1,7,642.48,642.10
7,1,8,642.56,642.48
8,1,9,642.12,642.56
9,1,10,641.71,642.12


#### Previous-Cycle Value — Lag Feature

A lag feature stores the sensor value from the previous cycle.

It provides the model with immediate historical information.

The first cycle of each engine has no previous observation and therefore receives NaN.

In [13]:
train_df["sensor_2_diff1"] = (
    train_df.groupby("unit")["sensor_2"]
    .diff()
)

In [14]:
train_df["sensor_2_diff1"] = (
    train_df.groupby("unit")["sensor_2"]
    .diff()
)

##### One-Cycle Difference

The difference feature measures the change in a sensor between the current and previous cycle.

Positive values indicate an increase, negative values indicate a decrease, and values near zero indicate little change.

In [15]:
diff_cols = []

for sensor in sensor_cols:

    col = f"{sensor}_diff1"

    train_df[col] = (
        train_df.groupby("unit")[sensor]
        .diff()
    )

    diff_cols.append(col)

print("Difference features:", len(diff_cols))

Difference features: 21


#### Extend Differences to All Sensors

After validating the difference calculation on one sensor, the same transformation is applied to all 21 sensors.

Each difference is calculated independently within each engine.

In [17]:
first_cycles = (
    train_df
    .groupby("unit")
    .head(1)
)

print(
    "First-cycle differences that are not NaN:",
    first_cycles[diff_cols].notna().sum().sum()
)

First-cycle differences that are not NaN: 0


#### Verify Engine Boundaries

The first cycle of every engine has no previous observation.

The boundary check confirms that temporal differences do not cross from one engine to another.

#### **Decision — Difference Features**

The one-cycle difference features are:

- chronologically correct,
- engine-specific,
- leakage-safe, and
- representative of recent sensor change.

They will be retained as candidate temporal features.

Their predictive usefulness will be tested later.

In [18]:
WINDOW = 5

train_df["sensor_2_rollmean5"] = (
    train_df.groupby("unit")["sensor_2"]
    .transform(
        lambda x: x.rolling(
            WINDOW,
            min_periods=1
        ).mean()
    )
)

In [19]:
train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_rollmean5"]
].head(10)

,unit,cycle,sensor_2,sensor_2_rollmean5
0,1,1,641.82,641.820000
1,1,2,642.15,641.985000
2,1,3,642.35,642.106667
3,1,4,642.35,642.167500
4,1,5,642.37,642.208000
5,1,6,642.10,642.264000
6,1,7,642.48,642.330000
7,1,8,642.56,642.372000
8,1,9,642.12,642.326000
9,1,10,641.71,642.194000


#### Five-Cycle Rolling Mean

The rolling mean summarizes the recent sensor level using the current cycle and preceding cycles.

A five-cycle window is used to reduce short-term noise while preserving recent temporal information.

In [21]:
rollmean_cols = []

for sensor in sensor_cols:

    col = f"{sensor}_rollmean5"

    train_df[col] = (
        train_df.groupby("unit")[sensor]
        .transform(
            lambda x: x.rolling(
                WINDOW,
                min_periods=1
            ).mean()
        )
    )

    rollmean_cols.append(col)

print("Rolling mean features:", len(rollmean_cols))

Rolling mean features: 21


#### Extend Rolling Mean to All Sensors

The five-cycle rolling mean is applied to all 21 sensors while maintaining engine boundaries.

#### Rolling-Window Leakage Check

The rolling mean uses only the current and previous four cycles.

Future observations are never included.

Therefore, the feature can be constructed using information available at prediction time.

#### Decision — Rolling Mean

The five-cycle rolling mean provides a smoother representation of recent sensor behavior.

It is leakage-safe and engine-specific, so it will be retained as a candidate temporal feature family.

In [22]:
train_df["sensor_2_rollstd5"] = (
    train_df.groupby("unit")["sensor_2"]
    .transform(
        lambda x: x.rolling(
            WINDOW,
            min_periods=2
        ).std()
    )
)

In [23]:
train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_rollstd5"]
].head(10)

,unit,cycle,sensor_2,sensor_2_rollstd5
0,1,1,641.82,NaN
1,1,2,642.15,0.233345
2,1,3,642.35,0.267644
3,1,4,642.35,0.250117
4,1,5,642.37,0.234776
5,1,6,642.10,0.128374
6,1,7,642.48,0.139463
7,1,8,642.56,0.174270
8,1,9,642.12,0.208519
9,1,10,641.71,0.340705


#### Five-Cycle Rolling Variability

Rolling standard deviation measures how much the sensor has fluctuated during the recent five cycles.

It complements the rolling mean by describing recent sensor variability rather than sensor level.

In [24]:
rollstd_cols = []

for sensor in sensor_cols:

    col = f"{sensor}_rollstd5"

    train_df[col] = (
        train_df.groupby("unit")[sensor]
        .transform(
            lambda x: x.rolling(
                WINDOW,
                min_periods=2
            ).std()
        )
    )

    rollstd_cols.append(col)

print("Rolling std features:", len(rollstd_cols))

Rolling std features: 21


#### Extend Rolling Variability to All Sensors

The five-cycle rolling standard deviation is calculated for all 21 sensors within each engine.

In [25]:
print(
    "First-cycle rolling std missing:",
    train_df.groupby("unit")
    .head(1)[rollstd_cols]
    .notna()
    .sum()
    .sum()
)

First-cycle rolling std missing: 0


#### Validate Rolling Standard Deviation

The initial cycles contain expected missing values because insufficient historical observations are available to calculate variability.

This is a structural property of the feature, not a data-quality problem.

#### Decision — Rolling Standard Deviation

Rolling standard deviation captures recent sensor variability and does not use future information.

It will be retained as a candidate temporal feature family.

In [26]:
import numpy as np

def rolling_slope(x):

    y = np.asarray(x)
    t = np.arange(len(y))

    if len(y) < 2:
        return np.nan

    return np.polyfit(t, y, 1)[0]

In [28]:
train_df["sensor_2_slope5"] = (
    train_df.groupby("unit")["sensor_2"]
    .transform(
        lambda x: x.rolling(
            WINDOW,
            min_periods=2
        ).apply(
            rolling_slope,
            raw=True
        )
    )
)

In [29]:
train_df[
    ["unit", "cycle", "sensor_2", "sensor_2_slope5"]
].head(10)

,unit,cycle,sensor_2,sensor_2_slope5
0,1,1,641.82,NaN
1,1,2,642.15,0.330
2,1,3,642.35,0.265
3,1,4,642.35,0.179
4,1,5,642.37,0.130
5,1,6,642.10,-0.008
6,1,7,642.48,0.001
7,1,8,642.56,0.053
8,1,9,642.12,-0.004
9,1,10,641.71,-0.114


#### Five-Cycle Local Trend

The local slope summarizes the recent direction of sensor movement.

Positive slope indicates an increasing trend.

Negative slope indicates a decreasing trend.

A value near zero indicates little directional movement.

In [30]:
slope_cols = []

for sensor in sensor_cols:

    col = f"{sensor}_slope5"

    train_df[col] = (
        train_df.groupby("unit")[sensor]
        .transform(
            lambda x: x.rolling(
                WINDOW,
                min_periods=2
            ).apply(
                rolling_slope,
                raw=True
            )
        )
    )

    slope_cols.append(col)

print("Slope features:", len(slope_cols))

C:\Users\User\AppData\Local\Temp\ipykernel_17376\3114145206.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col] = (


Slope features: 21


#### Extend Local Trend to All Sensors

The five-cycle local slope is calculated for all 21 sensors while preserving engine boundaries.

In [31]:
diff_cols = [
    f"{sensor}_diff1"
    for sensor in sensor_cols
]

rollmean_cols = [
    f"{sensor}_rollmean5"
    for sensor in sensor_cols
]

rollstd_cols = [
    f"{sensor}_rollstd5"
    for sensor in sensor_cols
]

slope_cols = [
    f"{sensor}_slope5"
    for sensor in sensor_cols
]

#### Organize Temporal Feature Families

The generated temporal features are organized into separate families so that each family can be evaluated independently.

In [32]:
print("Raw:", len(sensor_cols))
print("Difference:", len(diff_cols))
print("Rolling mean:", len(rollmean_cols))
print("Rolling std:", len(rollstd_cols))
print("Slope:", len(slope_cols))

Raw: 21
Difference: 21
Rolling mean: 21
Rolling std: 21
Slope: 21


#### Candidate Feature Count

Five feature families were investigated across 21 sensors, producing up to 105 candidate sensor-derived features.

This is a candidate pool, not automatically the final feature set.

In [33]:
candidate_cols = (
    sensor_cols
    + diff_cols
    + rollmean_cols
    + rollstd_cols
    + slope_cols
)

constant_cols = [
    col
    for col in candidate_cols
    if train_df[col].dropna().nunique() <= 1
]

print("Constant features:", constant_cols)

Constant features: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19', 'sensor_1_diff1', 'sensor_5_diff1', 'sensor_10_diff1', 'sensor_16_diff1', 'sensor_18_diff1', 'sensor_19_diff1', 'sensor_1_rollmean5', 'sensor_5_rollmean5', 'sensor_10_rollmean5', 'sensor_16_rollmean5', 'sensor_18_rollmean5', 'sensor_19_rollmean5', 'sensor_1_rollstd5', 'sensor_5_rollstd5', 'sensor_10_rollstd5', 'sensor_16_rollstd5', 'sensor_18_rollstd5', 'sensor_19_rollstd5']


#### Remove Constant Features

Features with no variation cannot provide useful information for distinguishing observations.

Such features are excluded from the candidate feature set.

In [34]:
final_candidate_cols = [
    col
    for col in candidate_cols
    if col not in constant_cols
]

phase2_df = train_df[
    ["unit", "cycle", "RUL"] + final_candidate_cols
].copy()

#### Create Candidate Phase 2 Dataset

The candidate dataset combines the RUL target with the original and temporal sensor features that passed the basic feature-quality checks.

In [35]:
print(
    "Total missing values:",
    phase2_df[final_candidate_cols].isna().sum().sum()
)

Total missing values: 5100


#### Inspect Temporal Missing Values

Missing values at the beginning of engine trajectories are expected because temporal features require historical observations.

They will be handled consistently before model training.

#### Leakage Check

All Phase 2 temporal features use information available at or before the current cycle.

No future sensor measurements are used.

All temporal operations are grouped by engine.

Therefore, the feature construction respects the prediction-time information boundary.

#### Leakage Check

All Phase 2 temporal features use information available at or before the current cycle.

No future sensor measurements are used.

All temporal operations are grouped by engine.

Therefore, the feature construction respects the prediction-time information boundary.

#### Final Chronological Check

The final Phase 2 dataset remains chronologically ordered within every engine.

This confirms that the temporal structure was preserved during feature construction.

In [36]:
print(
    "Duplicate unit-cycle rows:",
    phase2_df.duplicated(
        ["unit", "cycle"]
    ).sum()
)

Duplicate unit-cycle rows: 0


##### Duplicate Check

Each engine-cycle observation appears only once in the dataset.

In [37]:
output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "FD001_temporal_features.csv"
)

phase2_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: c:\Users\User\Desktop\jet-engine-predictive-maintenance\data\processed\FD001_temporal_features.csv


#### Save Temporal Feature Dataset

The Phase 2 candidate feature dataset is saved for reproducible use in the next stage.

####  Conclusion

Temporal Features transformed historical sensor measurements into temporal feature candidates.

The investigated feature families were:

- Raw sensor values
- Previous-cycle values
- One-cycle differences
- Five-cycle rolling means
- Five-cycle rolling standard deviations
- Five-cycle local slopes

All temporal operations were constructed within individual engine trajectories and without using future observations.

The resulting dataset provides a leakage-safe temporal representation for the next stage.

##### Important

The temporal features are candidates.

Their actual predictive usefulness has not yet been proven.

The next phase will compare these feature sets against the raw-sensor baseline using model validation.